# DDQN + Multi-Head Attention (Dot-Product) on MiniGrid-DoorKey-5x5

对照实验: 应用一组已知比较健壮的默认值, 用于隔离"到底是哪个组件"导致 `Multihead.ipynb` 里的 Q 值坍缩.

## 相比 `Multihead.ipynb` 的改动

1. **Attention: additive → 缩放点积 (scaled dot-product).**
   `A = Q @ K.transpose(-1,-2) / sqrt(d); softmax(dim=-1)`. `A[f, c]` 现在是 query 节点 f 与 key 节点 c 之间的相似度, 语义明确.
2. **Q 输出去掉 `elu`.** 保持线性, 避免把 Q 值挤到 (-1, ∞).
3. **`update_replay` 不再做 `N=50` 倍复制.** 每条经验只存一份.
4. **ε 从 1.0 线性退火到 0.05, 10k 步.**
5. **`lr = 1e-4` (原来 5e-4)**, `update_freq = 500` (原来 100).
6. **单次干净的网络初始化** (原来 notebook 里出现过重复 `GWagent = ...` 的问题, 这里避免).

## 唯一保持不变的

- 环境: `MiniGrid-DoorKey-5x5-v0` + `ImgObsWrapper` (partial obs, 7×7 egocentric)
- 网络其余结构: 两层 1x1 conv, 3 heads, node_size=64, spatial coords, LayerNorm, max pool, linear head
- DDQN 目标计算逻辑

如果这个 notebook 训练出可用的策略, 我们能确认原来至少有一个 (或组合) 是坍缩原因. 如果**仍然坍缩**, 说明问题在 max pool / norm1 / 部分可观测这些更深处, 需要下一轮实验.

In [ ]:
from einops import rearrange
import torch
from torch import nn
import numpy as np
import time
from collections import deque

import gymnasium as gym
from minigrid.wrappers import ImgObsWrapper

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

ENV_ID = 'MiniGrid-DoorKey-5x5-v0'
OBS_SIZE = 7   # ImgObsWrapper 输出 7×7

## 网络: 点积注意力版

只有 forward 里 attention 逻辑改了; 其他与 `Multihead.ipynb` 相同.

In [ ]:
class MultiHeadRelationalModule(torch.nn.Module):
    def __init__(self, obs_size=7):
        super().__init__()
        self.conv1_ch = 16
        self.conv2_ch = 20
        self.node_size = 64
        self.out_dim = 5
        self.ch_in = 3
        self.sp_coord_dim = 2
        self.N = obs_size * obs_size
        self.n_heads = 3

        self.conv1 = nn.Conv2d(self.ch_in, self.conv1_ch, kernel_size=(1,1), padding=0)
        self.conv2 = nn.Conv2d(self.conv1_ch, self.conv2_ch, kernel_size=(1,1), padding=0)

        self.proj_shape = (self.conv2_ch + self.sp_coord_dim, self.n_heads * self.node_size)
        self.k_proj = nn.Linear(*self.proj_shape)
        self.q_proj = nn.Linear(*self.proj_shape)
        self.v_proj = nn.Linear(*self.proj_shape)

        self.node_shape = (self.n_heads, self.N, self.node_size)
        self.k_norm = nn.LayerNorm(self.node_shape, elementwise_affine=True)
        self.q_norm = nn.LayerNorm(self.node_shape, elementwise_affine=True)
        self.v_norm = nn.LayerNorm(self.node_shape, elementwise_affine=True)

        self.linear1 = nn.Linear(self.n_heads * self.node_size, self.node_size)
        self.norm1 = nn.LayerNorm([self.N, self.node_size], elementwise_affine=True)
        self.linear2 = nn.Linear(self.node_size, self.out_dim)

    def forward(self, x):
        B, Cin, H, W = x.shape
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        with torch.no_grad():
            self.conv_map = x.clone()

        _, _, cH, cW = x.shape
        xcoords = torch.arange(cW, device=x.device).repeat(cH, 1).float() / cW
        ycoords = torch.arange(cH, device=x.device).repeat(cW, 1).transpose(1, 0).float() / cH
        spatial_coords = torch.stack([xcoords, ycoords], dim=0).unsqueeze(0).repeat(B, 1, 1, 1)

        x = torch.cat([x, spatial_coords], dim=1)
        x = x.permute(0, 2, 3, 1).flatten(1, 2)  # (B, N, 22)

        K = rearrange(self.k_proj(x), 'b n (head d) -> b head n d', head=self.n_heads)
        Q = rearrange(self.q_proj(x), 'b n (head d) -> b head n d', head=self.n_heads)
        V = rearrange(self.v_proj(x), 'b n (head d) -> b head n d', head=self.n_heads)
        K = self.k_norm(K)
        Q = self.q_norm(Q)
        V = self.v_norm(V)

        # 缩放点积: A[b, h, f, c] = < Q[b, h, f, :], K[b, h, c, :] > / sqrt(d)
        # 语义: query 节点 f 与 key 节点 c 的相似度
        A = torch.einsum('bhnd,bhmd->bhnm', Q, K)
        A = A / np.sqrt(self.node_size)
        A = torch.nn.functional.softmax(A, dim=-1)  # 对 key 节点归一化
        with torch.no_grad():
            self.att_map = A.clone()

        E = torch.einsum('bhnm,bhmd->bhnd', A, V)   # 从 key 节点聚合信息
        E = rearrange(E, 'b head n d -> b n (head d)')
        E = torch.relu(self.linear1(E))
        E = self.norm1(E)
        E = E.max(dim=1)[0]                          # 保留 max pool 用于隔离变量

        # 线性 Q 头, 不加 elu
        y = self.linear2(E)
        return y

## Helpers

In [ ]:
def prepare_state(x: np.ndarray):
    # (W, H, C) -> (C, W, H)
    ns = torch.from_numpy(x).float().permute(2, 0, 1).unsqueeze(dim=0)
    return ns.to(device)


def get_minibatch(replay, size):
    batch_ids = np.random.randint(0, len(replay), size)
    batch = [replay[i] for i in batch_ids]
    state_batch = torch.cat([e[0] for e in batch]).to(device)
    action_batch = torch.tensor([e[1] for e in batch], dtype=torch.long, device=device)
    reward_batch = torch.tensor([e[2] for e in batch], dtype=torch.float32, device=device)
    state2_batch = torch.cat([e[3] for e in batch]).to(device)
    done_batch = torch.tensor([e[4] for e in batch], dtype=torch.float32, device=device)
    return state_batch, action_batch, reward_batch, state2_batch, done_batch


def get_qtarget_ddqn(qvals, r, df, done):
    return r + (1 - done) * df * qvals


def lossfn(pred, targets, actions):
    q_sa = pred.gather(dim=1, index=actions.unsqueeze(dim=1))
    return torch.mean(torch.pow(targets.detach() - q_sa.squeeze(), 2))


# 不再做正奖励复制, 每条经验存一份
def update_replay(replay, exp, replay_size):
    replay.append(exp)
    return replay


action_map = {0: 0, 1: 1, 2: 2, 3: 3, 4: 5}


def epsilon(step, eps_start=1.0, eps_end=0.05, eps_decay=10000):
    return max(eps_end, eps_start - (eps_start - eps_end) * step / eps_decay)

In [ ]:
# 快速通网测试, 确认没有形状问题
test_env = ImgObsWrapper(gym.make(ENV_ID))
obs, _ = test_env.reset(seed=0)
print('obs shape:', obs.shape)
test_env.close()

m = MultiHeadRelationalModule(obs_size=OBS_SIZE).to(device)
with torch.no_grad():
    dummy = torch.randn(1, 3, OBS_SIZE, OBS_SIZE, device=device)
    out = m(dummy)
print('model output shape:', out.shape, '(expected torch.Size([1, 5]))')
print('att_map shape:     ', m.att_map.shape, f'(expected torch.Size([1, 3, {OBS_SIZE*OBS_SIZE}, {OBS_SIZE*OBS_SIZE}]))')

# 检查初始化的 Q spread 是否合理 (~0.5+, 不应该一开始就坍缩)
with torch.no_grad():
    q_samples = torch.stack([m(torch.randn(1, 3, OBS_SIZE, OBS_SIZE, device=device)).squeeze() for _ in range(10)])
spreads = (q_samples.max(dim=1).values - q_samples.min(dim=1).values).cpu().numpy()
print(f'\ninit Q spreads on random inputs: mean={spreads.mean():.3f} min={spreads.min():.3f} max={spreads.max():.3f}')
print('(健康的网络应该 mean spread > 0.1; 太小说明还没训就已坍缩)')

## 训练

**关键变化**: ε annealing, lr=1e-4, update_freq=500, 无正奖励复制.

In [ ]:
losses = []
epochs_length = []
eps_history = []

DEBUG_UNTIL = 20
DEBUG_EVERY = 500

env = ImgObsWrapper(gym.make(ENV_ID))
state = prepare_state(env.reset()[0])

# 干净的单次初始化
GWagent = MultiHeadRelationalModule(obs_size=OBS_SIZE).to(device)
Tnet = MultiHeadRelationalModule(obs_size=OBS_SIZE).to(device)
Tnet.load_state_dict(GWagent.state_dict())

maxsteps = 400
env.unwrapped.max_steps = maxsteps

epochs = 30000
replay_size = 9000
lr = 1e-4               # 原来 5e-4
gamma = 0.99
batch_size = 64

replay = deque(maxlen=replay_size)
opt = torch.optim.Adam(GWagent.parameters(), lr=lr)

update_freq = 500        # 原来 100

t0 = time.time()
eps_len = 0

for i in range(epochs):
    eps_len += 1
    eps = epsilon(i)
    eps_history.append(eps)

    with torch.no_grad():
        pred = GWagent(state)
    action = int(torch.argmax(pred))
    if np.random.rand() < eps:
        action = int(torch.randint(0, 5, size=(1,)).squeeze())
    action_d = action_map[action]

    state2, reward, terminated, truncated, _ = env.step(action_d)
    state2 = prepare_state(state2)
    reward = -0.01 if reward == 0 else reward
    done = bool(terminated or truncated)
    exp = (state, action, reward, state2, done)
    replay = update_replay(replay, exp, replay_size)

    if i < DEBUG_UNTIL or i % DEBUG_EVERY == 0:
        q_np = pred.squeeze().cpu().numpy()
        spread = q_np.max() - q_np.min()
        print(f'i={i:5d} eps={eps:.2f} | Q={q_np.round(3)} spread={spread:.4f} '
              f'| a={action} r={reward:+.2f} term={terminated} trunc={truncated}')

    if done:
        state = prepare_state(env.reset()[0])
        env.unwrapped.max_steps = maxsteps
        epochs_length.append(eps_len)
        eps_len = 0
    else:
        state = state2

    if len(replay) > batch_size:
        opt.zero_grad()
        sb, ab, rb, s2b, db = get_minibatch(replay, batch_size)
        q_pred = GWagent(sb)
        astar = torch.argmax(q_pred, dim=1)
        qs = Tnet(s2b).gather(dim=1, index=astar.unsqueeze(dim=1)).squeeze()
        targets = get_qtarget_ddqn(qs.detach(), rb, gamma, db)
        loss = lossfn(q_pred, targets.detach(), ab)
        losses.append(loss.detach().cpu().item())

        if i < DEBUG_UNTIL or i % DEBUG_EVERY == 0:
            print(f'       train | loss={loss.item():.4f} '
                  f'| target mean={targets.mean().item():.3f} std={targets.std().item():.3f} '
                  f'| q_pred mean={q_pred.mean().item():.3f} std={q_pred.std().item():.3f} '
                  f'| rb mean={rb.mean().item():+.3f}')

        loss.backward()
        torch.nn.utils.clip_grad_norm_(GWagent.parameters(), max_norm=1.0)
        opt.step()

        if i % update_freq == 0:
            Tnet.load_state_dict(GWagent.state_dict())

    if (i + 1) % 2000 == 0:
        elapsed = time.time() - t0
        eta = elapsed / (i + 1) * (epochs - i - 1)
        recent = losses[-100:] if losses else []
        avg = sum(recent) / len(recent) if recent else float('nan')
        completed = len(epochs_length)
        recent_wins = sum(1 for L in epochs_length[-20:] if L < 400)
        print(f'epoch {i+1}/{epochs} | elapsed {elapsed/60:.1f}min | ETA {eta/60:.1f}min '
              f'| loss(avg100) {avg:.4f} | eps={eps:.2f} | episodes={completed} wins_last20={recent_wins}/20')

print(f'\ndone in {(time.time()-t0)/60:.1f} min')
print(f'positive-reward transitions in replay: {sum(1 for e in replay if e[2] > 0)} / {len(replay)}')
print(f'episodes completed: {len(epochs_length)}, min length: {min(epochs_length) if epochs_length else "n/a"}')
wins = [L for L in epochs_length if L < 400]
print(f'wins (natural termination): {len(wins)} / {len(epochs_length)}')

## 评估

In [ ]:
GWagent.eval()
n_eval = 50
successes = 0
step_counts = []
returns = []

with torch.no_grad():
    for ep in range(n_eval):
        s = prepare_state(env.reset()[0])
        ep_return, steps, done = 0.0, 0, False
        while not done:
            q = GWagent(s)
            a = int(torch.argmax(q))
            s2, r, term, trunc, _ = env.step(action_map[a])
            ep_return += r
            steps += 1
            done = term or trunc
            s = prepare_state(s2)
        if term and ep_return > 0:
            successes += 1
        step_counts.append(steps)
        returns.append(ep_return)

print(f'success rate (greedy): {successes}/{n_eval} = {successes/n_eval:.1%}')
print(f'avg steps:             {np.mean(step_counts):.1f}')
print(f'avg return:            {np.mean(returns):.3f}')
GWagent.train()

In [ ]:
# 诊断: 网络对不同输入是否产生不同 Q
import random
GWagent.eval()
print('=== Q-values across 5 replay states ===')
with torch.no_grad():
    for j, exp in enumerate(random.sample(list(replay), min(5, len(replay)))):
        s = exp[0]
        q = GWagent(s).squeeze().cpu().numpy()
        print(f'state {j}: Q={q.round(4)} spread={q.max()-q.min():.4f}')

print('\n=== synthetic inputs ===')
with torch.no_grad():
    z = torch.zeros(1, 3, OBS_SIZE, OBS_SIZE, device=device)
    o = torch.ones(1, 3, OBS_SIZE, OBS_SIZE, device=device) * 5
    r = torch.rand(1, 3, OBS_SIZE, OBS_SIZE, device=device) * 10
    print(f'zeros:  Q={GWagent(z).squeeze().cpu().numpy().round(4)}')
    print(f'fives:  Q={GWagent(o).squeeze().cpu().numpy().round(4)}')
    print(f'random: Q={GWagent(r).squeeze().cpu().numpy().round(4)}')
GWagent.train()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
if losses:
    axes[0].plot(np.log(np.clip(np.array(losses), 1e-8, None)))
    axes[0].set_title('log(loss)')
    axes[0].set_xlabel('gradient step')
if epochs_length:
    axes[1].plot(epochs_length)
    axes[1].axhline(400, color='red', linestyle='--', label='timeout')
    axes[1].set_title('episode length')
    axes[1].set_xlabel('episode')
    axes[1].legend()
if eps_history:
    axes[2].plot(eps_history)
    axes[2].set_title('epsilon schedule')
    axes[2].set_xlabel('epoch')
plt.tight_layout()
plt.show()